# Wan 2.2 Img2Img — Фото из фото**Генерация стилизованных изображений** с помощью Wan 2.2 14B + Lightning LoRA (4 шага!) + апскейл x2.**Время настройки:** ~10-15 мин (скачивание моделей ~15 ГБ)**VRAM:** ~14 ГБ пиковое потребление (T4 16 ГБ)---### Как это работает```[1. GPU]  →  [2. Установка]  →  [3. Ноды]  →  [4. Модели]  →  [5. Воркфлоу]  →  [6. ЗАГРУЗКА ФОТО]  →  [7. Запуск]                                                                                          ↑                                                                                 Самый важный шаг!                                                                        Загрузите исходное фото для                                                                        стилизации (любое фото/арт)```---### Что делает этот ноутбук?| Возможность | Описание ||-------------|----------|| **Img2Img** | Загрузите любое фото → получите стилизованную версию || **Lightning (4 шага)** | Генерация за 4 шага вместо 20-30, в 5-7 раз быстрее || **Апскейл x2** | Встроенный UltimateSDUpscale: 1280x1920 → 2560x3840 || **Гибкий denoise** | 0.3 = лёгкая стилизация, 0.7 = сильная переработка |---### Требования к входному фото| Параметр | Рекомендация ||----------|-------------|| **Формат** | .jpg, .png || **Размер** | Любой (будет ресайзнуто до 1280x1920) || **Качество** | Чёткое, без водяных знаков || **Тип** | Портреты, пейзажи, арт — всё подходит |### Чего НЕ нужно- Видео или аудио файлы- GIF-анимации- Файлы больше 20 МБ

In [ ]:
#@title 1. Проверка GPU!nvidia-smi --query-gpu=name,memory.total --format=csv,noheaderimport torchprint(f"CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI%cd /content!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже склонировано"%cd /content/ComfyUI!pip install -r requirements.txt -qprint("\nComfyUI установлен!")

In [ ]:
#@title 3. Установка кастомных нод%cd /content/ComfyUI/custom_nodes# Основные нодыrepos = {    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",    "ComfyUI-mxToolkit": "https://github.com/Smirnov75/ComfyUI-mxToolkit.git",    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",    "ComfyUI-Custom-Scripts": "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",    "ComfyUI-wanBlockswap": "https://github.com/orssorbit/ComfyUI-wanBlockswap.git",    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",    "ComfyUI_UltimateSDUpscale": "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git",    "ComfyUI-Image-Saver": "https://github.com/alexopus/ComfyUI-Image-Saver.git",}for name, url in repos.items():    !git clone {url} 2>/dev/null || echo "{name} уже склонирован"# Зависимостиimport osfor name in repos:    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"    if os.path.exists(req):        !pip install -r {req} -qprint(f"\n{len(repos)} кастомных нод установлено!")

In [ ]:
#@title 4. Скачивание моделей (~15 ГБ, 5-10 мин)import osM = "/content/ComfyUI/models"os.makedirs(f"{M}/diffusion_models/WAN", exist_ok=True)os.makedirs(f"{M}/text_encoders", exist_ok=True)os.makedirs(f"{M}/vae", exist_ok=True)os.makedirs(f"{M}/upscale_models", exist_ok=True)os.makedirs(f"{M}/loras/WAN", exist_ok=True)# Comfy-Org модели (нативный формат)COMFY22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"COMFY21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"KIJAI = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"files = {    # Wan 2.2 T2V Low Noise 14B fp8 (~9.5 ГБ)    f"{M}/diffusion_models/WAN/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors":        f"{COMFY22}/diffusion_models/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors",    # Текстовый энкодер UMT5 XXL fp8 (~5 ГБ)    f"{M}/text_encoders/umt5-xxl-encoder-fp8-e4m3fn-scaled.safetensors":        f"{COMFY22}/text_encoders/umt5-xxl-encoder-fp8-e4m3fn-scaled.safetensors",    # VAE (~200 МБ)    f"{M}/vae/wan_2.1_vae.safetensors":        f"{COMFY21}/vae/wan_2.1_vae.safetensors",    # Апскейлер 4x ClearReality (~64 МБ)    f"{M}/upscale_models/4x-ClearRealityV1.pth":        "https://huggingface.co/Kim2091/ClearRealityV1/resolve/main/4x-ClearRealityV1.pth",    # Lightning LoRA — 4 шага вместо 20 (~614 МБ)    f"{M}/loras/WAN/Wan2.2-Lightning_T2V-A14B-4steps-lora_LOW_fp16.safetensors":        f"{KIJAI}/Wan22-Lightning/Wan2.2-Lightning_T2V-v1.1-A14B-4steps-lora_LOW_fp16.safetensors",}for dest, url in files.items():    if os.path.exists(dest):        print(f"Уже существует: {os.path.basename(dest)}")    else:        print(f"\nСкачивание {os.path.basename(dest)}...")        !wget -q --show-progress -O "{dest}" "{url}"print("\nВсе модели готовы!")!du -sh {M}/diffusion_models/WAN/* {M}/text_encoders/* {M}/vae/* {M}/upscale_models/* {M}/loras/WAN/*

In [ ]:
#@title 5. Скачивание воркфлоуimport osGIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"WF_DIR = "/content/ComfyUI/user/default/workflows"os.makedirs(WF_DIR, exist_ok=True)!wget -q -O "{WF_DIR}/photo_wan_img2img.json" "{RAW}/photo_wan_img2img.json"print("Скачано: photo_wan_img2img.json")print(f"\nВоркфлоу сохранён в {WF_DIR}")

In [ ]:
#@title 6. Загрузка исходного фото#@markdown ## Нажмите кнопку "Выбрать файлы" ниже#@markdown#@markdown Загрузите фото, которое хотите стилизовать.#@markdown Файл сохранится в `/content/ComfyUI/input/`.#@markdown#@markdown ---#@markdown#@markdown ### Какие фото подходят?#@markdown#@markdown | Тип | Подходит? | Пример |#@markdown |-----|-----------|--------|#@markdown | Портрет | Да | Фото лица, поясной портрет |#@markdown | Пейзаж | Да | Природа, городские виды |#@markdown | Арт/иллюстрация | Да | Рисунки, цифровой арт |#@markdown | Фото с текстом | Нет | Скриншоты, мемы |from google.colab import filesfrom IPython.display import display, Image as IPImageimport osINPUT_DIR = "/content/ComfyUI/input"os.makedirs(INPUT_DIR, exist_ok=True)VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')print("=" * 50)print("  ЗАГРУЗКА ФОТО ДЛЯ IMG2IMG")print("=" * 50)print(f"  Папка: {INPUT_DIR}")print(f"  Форматы: {', '.join(VALID_EXT)}")print()uploaded = files.upload()saved = []skipped = []for name, data in uploaded.items():    ext = os.path.splitext(name)[1].lower()    if ext not in VALID_EXT:        skipped.append(name)        continue    path = os.path.join(INPUT_DIR, name)    with open(path, "wb") as f:        f.write(data)    saved.append(name)print(f"\n{'=' * 50}")print("  РЕЗУЛЬТАТ")print(f"{'=' * 50}")if saved:    print(f"\n  Загружено: {len(saved)} файлов")    for f in saved:        print(f"    {f}")if skipped:    print(f"\n  Пропущено (неподдерживаемый формат): {len(skipped)}")    for f in skipped:        print(f"    {f}")# Превьюif saved:    print(f"\nПревью:")    for f in saved[:4]:        path = os.path.join(INPUT_DIR, f)        try:            display(IPImage(filename=path, width=300))        except Exception:            pass    print("\nОткройте ComfyUI, загрузите воркфлоу и выберите это фото в ноде LoadImage")

In [ ]:
#@title 7. Запуск ComfyUIimport subprocess, time, re# Cloudflared!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1# Запуск ComfyUIcomfy = subprocess.Popen(    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],    cwd="/content/ComfyUI",    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT)print("Запуск ComfyUI... (30с)")time.sleep(30)# Туннельtunnel = subprocess.Popen(    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT)time.sleep(5)url = Nonefor _ in range(20):    line = tunnel.stdout.readline().decode()    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)    if match:        url = match.group(0)        breakif url:    print(f"\n{'='*60}")    print(f"  ComfyUI готов!")    print(f"  Откройте: {url}")    print(f"{'='*60}")    print("\n  1. Меню → Load → photo_wan_img2img.json")    print("  2. В ноде LoadImage выберите загруженное фото")    print("  3. Напишите промпт (стиль, настроение, детали)")    print("  4. Нажмите Queue Prompt")    print("\n  Генерация: ~1-2 мин (4 шага Lightning)")    print("  Апскейл: +2-3 мин")else:    print("Ссылка не найдена. ComfyUI на порту 8188.")

In [ ]:
#@title 8. Сохранение на Google Drivefrom google.colab import driveimport shutil, glob, osdrive.mount('/content/drive')src = "/content/ComfyUI/output"dst = "/content/drive/MyDrive/ComfyUI_Output/img2img"os.makedirs(dst, exist_ok=True)results = glob.glob(f"{src}/**/*.png", recursive=True) + glob.glob(f"{src}/**/*.jpg", recursive=True)if not results:    print("Результатов пока нет. Сначала сгенерируйте изображение!")else:    for f in results:        shutil.copy2(f, dst)        print(f"Скопировано: {os.path.basename(f)}")    print(f"\n{len(results)} файлов сохранено в {dst}")

---## Гайд по воркфлоу `photo_wan_img2img`### Основные параметры| Параметр | Значение | Описание ||----------|----------|----------|| `denoise` | 0.3–0.9 | Сила изменений. 0.3 = лёгкая стилизация, 0.7 = средняя, 0.9 = почти полная перерисовка || `steps` | 4 | Lightning LoRA = 4 шага (не меняйте) || `CFG` | 1.0 | Для Lightning = 1.0 (не меняйте) || `NAG Scale` | 11 | Нормализованное внимание (можно оставить) || `resolution` | 1280x1920 | Портретная ориентация || `upscale_ratio` | 2x | 1280x1920 → 2560x3840 || `blocks_to_swap` | 40 | Для T4. Уменьшите для более быстрого GPU |### Примеры промптов```anime style illustration, vibrant colors, detailed background, studio ghibli aesthetic``````oil painting, impressionist style, thick brushstrokes, warm golden light``````cyberpunk photography, neon lights, rain reflections, cinematic composition```### Если получается OOM (нехватка памяти)1. Уменьшите разрешение в ноде ImageResize2. Отключите группу Upscale через Fast Groups Bypasser3. Увеличьте blocks_to_swap до 40 (максимум)